# LELA60331 Week 9 Seminar: Multilayer neural networks

The classifiers we have built so far (logistic regression and softmax) divide up the space of inputs with **straight** borders. This week we meet a very simple problem that no straight border can solve, the **exclusive or** (XOR) problem, and see how a network with an extra **hidden layer** solves it. We then work through how such a network is trained.

| Part | Topic |
|---|---|
| 1 | AND, OR and XOR |
| 2 | A network that solves XOR |
| 3 | The forward and backward pass, step by step |
| 4 | Training a network on XOR |

**Lectures:** this sheet builds on the Week 7 lecture, *Neural Networks 1* (the XOR problem and two-layer networks), and the Week 8 lecture, *Neural Networks 2: Training* (backpropagation). The network in Part 2 is the one from the Week 7 slides, and Part 3 follows the worked example in the Week 8 training lecture step by step.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(suppress=True, precision=3)

---
# Part 1 · AND, OR and XOR

Each of these three functions takes two inputs, each 0 or 1, and gives an output of 0 or 1:

| x1 | x2 | x1 AND x2 | x1 OR x2 | x1 XOR x2 |
|---|---|---|---|---|
| 0 | 0 | 0 | 0 | 0 |
| 0 | 1 | 0 | 1 | 1 |
| 1 | 0 | 0 | 1 | 1 |
| 1 | 1 | 1 | 1 | 0 |

* **AND** is 1 only when both inputs are 1.
* **OR** is 1 when at least one input is 1.
* **XOR** ("exclusive or") is 1 when **exactly one** input is 1.

Can logistic regression learn these? Here is a NumPy logistic regression function, using the bias trick from Week 8: each row of `X` has the two inputs plus an extra 1.

In [ ]:
def train_logistic(X, y, lr=0.5, n_iters=5000):
    """Train a logistic regression model and return its weights."""
    np.random.seed(1)
    weights = np.random.rand(X.shape[1])
    for i in range(n_iters):
        q = 1 / (1 + np.exp(-X.dot(weights)))
        weights = weights - lr * X.T.dot(q - y) / len(y)
    return weights

def predict_logistic(X, weights):
    return 1 / (1 + np.exp(-X.dot(weights)))

X = np.array([[0, 0, 1],
              [0, 1, 1],
              [1, 0, 1],
              [1, 1, 1]])

### 🏋️ Exercise 1 — Can logistic regression learn AND, OR and XOR?

Create the target outputs `y_and`, `y_or` and `y_xor` as NumPy arrays (one value for each row of `X`, in order), then run the cell. Which functions does logistic regression learn? What does it predict for XOR?

In [ ]:
y_and = ???
y_or = ???
y_xor = ???

for name, targets in [("AND", y_and), ("OR", y_or), ("XOR", y_xor)]:
    w = train_logistic(X, targets)
    print(name, "targets:", targets, " predictions:", predict_logistic(X, w))

The plot below shows why. For AND and OR, a straight line can separate the 1s (circles) from the 0s (crosses). For XOR, the two 1s are in opposite corners, and no single straight line can separate them from the two 0s.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, (name, targets) in zip(axes, [("AND", [0, 0, 0, 1]), ("OR", [0, 1, 1, 1]), ("XOR", [0, 1, 1, 0])]):
    for row, t in zip(X, targets):
        ax.scatter(row[0], row[1], marker="o" if t == 1 else "x", s=200)
    ax.set_title(name)
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(-0.5, 1.5)
plt.show()

---
# Part 2 · A network that solves XOR

A **multilayer network** puts a **hidden layer** of units between the inputs and the output:

```
 inputs (layer 0)        hidden units (layer 1)        output (layer 2)
   x1  ─┐                 ┌─ h1 ─┐
   x2  ─┼── weights_0_1 ──┤      ├── weights_1_2 ──  output
   1   ─┘                 └─ h2 ─┘
```

Each hidden unit computes a weighted sum of the inputs, exactly as in logistic regression, and then applies an **activation function**. We use the **ReLU** ("rectified linear unit"), which turns negative numbers into 0 and leaves positive numbers alone: `np.maximum(z, 0)`. The output is a weighted sum of the hidden units.

The activation function is what makes the hidden layer useful: without it, two layers of weighted sums would just add up to one weighted sum, and we would be back to a straight border.

Here are weights that solve XOR: the network from the Week 7 lecture slides. `weights_0_1` has one row per input (x1, x2 and the extra 1) and one column per hidden unit. `weights_1_2` has one weight per hidden unit.

In [ ]:
weights_0_1 = np.array([[1,  1],     # from x1 to h1 and h2
                        [1,  1],     # from x2 to h1 and h2
                        [0, -1]])    # from the extra 1 (the biases of h1 and h2)
weights_1_2 = np.array([1, -2])      # from h1 and h2 to the output

def relu(z):
    return np.maximum(z, 0)

print(relu(np.array([-2, -0.5, 0, 1.5, 3])))

### 🏋️ Exercise 2 — A forward pass through the network

For each row of `X`, calculate `hidden`, the values of the two hidden units (the ReLU of the row times `weights_0_1`), and `output`, the network's output (the hidden values times `weights_1_2`). You should find that the outputs are exactly the XOR targets.

In [ ]:
for row in X:
    hidden = ???
    output = ???
    print("input:", row[:2], " hidden:", hidden, " output:", output)

### 🏋️ Exercise 3 — How does it work?

Look at the hidden values you printed. Fill in the two descriptions below, choosing from `"at least one input is 1"`, `"both inputs are 1"` and `"neither input is 1"`. Then explain to your neighbour why `h1 - 2 * h2` gives XOR.

In [ ]:
h1_is_positive_when = ???
h2_is_positive_when = ???

---
# Part 3 · The forward and backward pass, step by step

Of course, we don't want to choose a network's weights by hand: we want it to **learn** them. This part follows the worked example in the Week 8 lecture, *Neural Networks 2: Training*. Here is a network with three hidden units and some starting weights, and a single training input `layer_0` whose correct output is 1. As before, `layer_0` ends with the extra 1. (We use the names `layer_0`, `layer_1` and `layer_2` for the input, the hidden units and the output.)

### 🏋️ Exercise 4 — The forward pass

Calculate `layer_1` (the hidden values) and `layer_2` (the output).

In [ ]:
weights_0_1 = np.array([[-0.59,  0.75, -0.95],
                        [ 0.34, -0.17,  0.12],
                        [-0.72, -0.60,  0.60]])
weights_1_2 = np.array([0.93, -0.37, 0.38])
layer_0 = np.array([0, 1, 1])
true_label = 1

layer_1 = ???
layer_2 = ???
print(layer_1, layer_2)           # you should get [0.   0.   0.72] and 0.2736

### The backward pass

The output, 0.27, should be 1. We measure the error with the squared difference between the output and the correct value, as in linear regression in Week 4. Let `diff = layer_2 - true_label`.

**Weights from the hidden layer to the output.** These work just like linear regression: the gradient for `weights_1_2[j]` is `diff * layer_1[j]`. So a hidden unit that was more active gets a bigger change.

**Weights from the inputs to the hidden layer.** For these we work **backwards** from the output (this is why it is called **backpropagation**). Each hidden unit's share of the blame is:

* `diff`, the output's error, times `weights_1_2[j]`, how strongly hidden unit `j` affects the output,
* times 1 if the hidden unit was active (`layer_1[j] > 0`), and 0 if the ReLU cut it off, because then changing its inputs a little would make no difference.

This share is the hidden unit's **error**. The gradient for `weights_0_1[i, j]` is then input `i` times hidden unit `j`'s error, which `np.outer` calculates for all `i` and `j` at once, just like the softmax gradient in Week 8.

### 🏋️ Exercise 5 — The backward pass and update

1. `diff`: the output minus the correct value
2. `hidden_error`: for each hidden unit, `diff * weights_1_2 * (layer_1 > 0)`
3. update `weights_1_2` and `weights_0_1` with a learning rate of 0.2

Which weights change, and which stay the same? Why?

In [ ]:
learning_rate = 0.2

diff = ???
hidden_error = ???
weights_1_2 = ???
weights_0_1 = ???
print(weights_0_1)
print(weights_1_2)
# you should get:
# [[-0.59   0.75  -0.95 ]
#  [ 0.34  -0.17   0.175]
#  [-0.72  -0.6    0.655]]
# [ 0.93  -0.37   0.485]

---
# Part 4 · Training a network on XOR

Now we put the forward pass, the backward pass and the update in a loop, and train the network on the four XOR examples. This network updates its weights after **each** example in turn, rather than once per pass through all the data.

### 🏋️ Exercise 6 — The training loop

Complete the forward pass, the backward pass and the updates, as in Exercises 4 and 5. The cell after it shows the trained network's output for each input, which should be close to 0, 1, 1, 0.

In [ ]:
inputs = np.array([[0, 0, 1],
                   [0, 1, 1],
                   [1, 0, 1],
                   [1, 1, 1]])
true_labels = np.array([0, 1, 1, 0])

learning_rate = 0.35
hidden_size = 3
np.random.seed(1)
weights_0_1 = np.random.rand(3, hidden_size)
np.random.seed(1)
weights_1_2 = np.random.rand(hidden_size)
loss = []
n_iters = 1000

for iteration in range(n_iters):
    total_error = 0
    for i in range(len(inputs)):
        layer_0 = inputs[i]
        # forward pass
        layer_1 = ???
        layer_2 = ???
        # backward pass
        diff = ???
        total_error += diff ** 2
        hidden_error = ???
        # update the weights
        weights_1_2 = ???
        weights_0_1 = ???
    loss.append(total_error)

plt.plot(range(1, n_iters), loss[1:])
plt.xlabel("number of epochs")
plt.ylabel("loss")
plt.show()

In [ ]:
for row in inputs:
    print(row[:2], round(float(relu(row.dot(weights_0_1)).dot(weights_1_2)), 3))

### 🏋️ Exercise 7 — The learning rate

Once your network makes good predictions, change the learning rate to 0.01 and train it again. What do you notice? Why do you think this happens? What happens if you also increase `n_iters`?

### 🏋️ Exercise 8 — Looking inside the trained network

Print the trained `weights_0_1` and `weights_1_2`, and the hidden values `layer_1` for each of the four inputs. Which hidden units are active for which inputs? Can you explain in words how the network computes XOR, as you did in Exercise 3?

In [ ]:
# Exercise 8

⭐ **Challenge 1 (optional):** change `np.random.seed(1)` to `np.random.seed(3)` in both places and train again. Look at the hidden values: what has gone wrong? (Hint: what does the ReLU do to a hidden unit whose weighted sum is negative for every input?)

⭐ **Challenge 2 (optional, harder):** rewrite the training loop so that each epoch processes all four inputs **at once**, using matrix operations instead of the inner `for` loop. `inputs.dot(weights_0_1)` gives the hidden values for every input at once. The gradients are then summed over the four inputs: for example, `layer_1.T.dot(diff)` for `weights_1_2`.

---
## ✏️ Summary

This week you have:

* seen that logistic regression can learn AND and OR, but not XOR, because no straight border separates XOR's classes
* used a **hidden layer** with the **ReLU** activation function to solve XOR
* carried out the **forward pass** and the **backward pass** (backpropagation) through a two-layer network
* trained a network on XOR, and looked at the effect of the learning rate